# Setting up the domains

The first step of every analysis is setting up the solid domains and their stiffness
properties. The library covers simply supported beams and plates, the supports are built
into the domains, so there is nothing more to say about them here.

## Euler-Bernoulli and Timoshenko beams


The library provides solutions for both Euler-Bernoully and Timoshenko beams, and setting up any of the two is quite similar. If you provide shear stiffness, the beam is a Timoshenko one, otherwise it is Euler-Bernoulli. The following code snippet illustrates the steps using the {class}`~sigmaepsilon.solid.fourier.beam.NavierBeam` class.

In [1]:
from sigmaepsilon.solid.fourier import NavierBeam

# geometry
beam_length = 1000.0
width, height = 20.0, 80.0  # rectangular cross-section

# material properties
young_modulus, poisson_ratio = 210000.0, 0.25  # material

# stiffness properties
inertia = width * height**3 / 12
bending_stiffness = young_modulus * inertia

# solution parameters
number_of_modes = 100

bernoulli_beam = NavierBeam(beam_length, number_of_modes, EI=bending_stiffness)

If your beam is of the Timoshenko type, you also have to provide the shear stiffness of the cross section.


In [2]:
# calculate shear stiffness
area = width * height
shear_modulus = young_modulus / (2 * (1 + poisson_ratio))
shear_correction_factor = 5 / 6
shear_stiffness = shear_modulus * area * shear_correction_factor

timoshenko_beam = NavierBeam(
    beam_length, number_of_modes, EI=bending_stiffness, GA=shear_stiffness
)

In both cases, you also had to specify the number of harmonic terms involved in the solution at creating the instances for your beams.


## Kirchhoff-Love and Uflyand-Mindlin plates


You can define Kirchhoff-Love or Uflyand-Mindlin plates using the 
{class}`~sigmaepsilon.solid.fourier.plate.NavierPlate` class. Things are a bit more complicated, 
as there are more stiffness parameters to calculate and both the bending and shear stiffness terms 
have to be provided with matrices, depending on the type of plate you want. If you know how to, you 
can calculate these stiffness matrices on your own, or you can use tools from other solutions from 
the SigmaEpsilon ecosystem. We definitively suggest you to utilize all that SigmaEpsilon has to offer.

First we create a Kirchhoff-Love plate. For this, you have to specify the material stiffness matrix for the plate, which, for a linearly isotropic material takes the form of

$$
D = \frac{E h^3}{12 (1 - \nu^2)}
\begin{bmatrix}
1 & \nu & 0 \\
\nu & 1 & 0 \\
0 & 0 & \frac{1 - \nu}{2}
\end{bmatrix}
$$

where $E$ is the Young's modulus, $\nu$ is the Poisson's ratio and $h$ is the thickness. Of course, this easy to do manually.


In [3]:
import numpy as np
from sigmaepsilon.solid.fourier import NavierPlate

# geometry
length_X, length_Y = (600.0, 800.0)
thickness = 25.0

# material properties
young_modulus = 2890.0
poisson_ratio = 0.2

# solution parameters
number_of_modes_X = 20
number_of_modes_Y = 20

# stiffness
D_factor = young_modulus * thickness**3 / (12 * (1 - poisson_ratio**2))
bending_stiffness = D_factor * np.array(
    [[1, poisson_ratio, 0], [poisson_ratio, 1, 0], [0, 0, (1 - poisson_ratio) / 2]]
)

kirchhoff_plate = NavierPlate(
    (length_X, length_Y),
    (number_of_modes_X, number_of_modes_Y),
    D=bending_stiffness,
)

If you want more power, you can use the tools from other libraries in the SigmaEpsilon ecosystem.


In [4]:
from sigmaepsilon.math.linalg import ReferenceFrame

from sigmaepsilon.solid.material import (
    KirchhoffPlateSection,
    ElasticityTensor,
    LinearElasticMaterial,
    HuberMisesHenckyFailureCriterion_SP,
)
from sigmaepsilon.solid.material.utils import elastic_stiffness_matrix

from sigmaepsilon.solid.fourier import NavierPlate

# GEOMETRY
length_X, length_Y = (600.0, 800.0)
thickness = 25.0

# MATERIAL PROPERTIES
young_modulus = 2890.0
poisson_ratio = 0.2
yield_strength = 2.0

# SOLUTION PARAMETERS
number_of_modes_X = 20
number_of_modes_Y = 20

# SETTING UP HOOKE'S LAW
hooke = elastic_stiffness_matrix(E=young_modulus, NU=poisson_ratio)
frame = ReferenceFrame(dim=3)
stiffness = ElasticityTensor(hooke, frame=frame, tensorial=False)
failure_model = HuberMisesHenckyFailureCriterion_SP(yield_strength=yield_strength)
material = LinearElasticMaterial(stiffness=stiffness, failure_model=failure_model)

# CALCULATING SECTION STIFFNESS
kirchhoff_section = KirchhoffPlateSection(
    layers=[
        KirchhoffPlateSection.Layer(material=material, thickness=thickness),
    ]
)
bending_stiffness = kirchhoff_section.elastic_stiffness_matrix()

kirchhoff_plate = NavierPlate(
    (length_X, length_Y),
    (number_of_modes_X, number_of_modes_Y),
    D=bending_stiffness,
)

Similarly to beams, if you want to set up an Uflyand-Mindlin plate, the only difference is that you also have to provide the shear stiffness terms at instantiation. These are described by the matrix

$$
G = \kappa \frac{E h}{2 (1 + \nu)}
\begin{bmatrix}
1 & 0 \\
0 & 1
\end{bmatrix}
$$

where $\kappa$ is the shear correction factor.


In [5]:
D_factor = young_modulus * thickness**3 / (12 * (1 - poisson_ratio**2))
bending_stiffness = D_factor * np.array(
    [[1, poisson_ratio, 0], [poisson_ratio, 1, 0], [0, 0, (1 - poisson_ratio) / 2]]
)
G_factor = (5 / 6) * young_modulus * thickness / (2 * (1 + poisson_ratio))
shear_stiffness = G_factor * np.array([[1, 0], [0, 1]])

mindlin_plate = NavierPlate(
    (length_X, length_Y),
    (number_of_modes_X, number_of_modes_Y),
    D=bending_stiffness,
    S=shear_stiffness,
)

Again, you can insted use the tools from the SigmaEpsilon ecosystem.


In [6]:
from sigmaepsilon.solid.material import MindlinPlateSection
from numpy import ascontiguousarray as ascont

mindlin_section = MindlinPlateSection(
    layers=[
        MindlinPlateSection.Layer(material=material, thickness=thickness),
    ]
)
ABDS_matrix = mindlin_section.elastic_stiffness_matrix()
bending_stiffness, shear_stiffness = (
    ascont(ABDS_matrix[:3, :3]),
    ascont(ABDS_matrix[3:, 3:]),
)

mindlin_plate = NavierPlate(
    (length_X, length_Y),
    (number_of_modes_X, number_of_modes_Y),
    D=bending_stiffness,
    S=shear_stiffness,
)

``{note}
This was a very basic setup for a plate. In more complex cases, it is recommended to
use the tools in [sigmaepsilon.solid.material](https://sigmaepsilonsolidmaterial.readthedocs.io/en/latest/?badge=latest) 
which, besides other things, calculates shear correction factors for anisotropic multi-layered plates as well.
``

## Next steps

With the domains at hand, the next step is
{doc}`specifying the boundary conditions <boundary_conditions>`.